# Установка зависимостей и проверка работы моделей(выполнять только один раз)

In [1]:
!pip install -q transformers datasets

In [2]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from google.colab import drive
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed
)
from transformers.trainer_utils import get_last_checkpoint

os.environ["SAFETENSORS_FAST_GPU"] = "0"

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

def set_seed(seed):
   random.seed(seed)
   np.random.seed(seed)
   torch.manual_seed(seed)
   torch.cuda.manual_seed_all(seed)

SEED = 42
set_seed(SEED)

device: cuda


In [4]:
from google.colab import drive

drive.mount('/content/drive')
drive_root = '/content/drive/MyDrive/papadyk-collab/vkr'

Mounted at /content/drive


In [5]:
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(HF_TOKEN)

In [9]:
import os
import gc
import math
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import balanced_accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    RobertaModel,
    RobertaConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

MODEL_NAME = "ai-forever/ruRoberta-large"

MAX_LENGTH = 512
STRIDE = 256
MAX_CHUNKS = 6

BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 20

LR_ENCODER = 1.2e-5
LR_HEAD = 3e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

HEAD_DROPOUT = 0.3
LABEL_SMOOTHING = 0.05

TEXT_COL = "text"
LABEL_COL = "label"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
class_weights_tensor = None


def compute_metrics(eval_pred):
    logits, labels_true = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "balanced_accuracy": balanced_accuracy_score(labels_true, preds),
        "f1_macro": f1_score(labels_true, preds, average="macro", zero_division=0),
    }


def tokenize_document(example):
    encoded = tokenizer(
        example[TEXT_COL],
        truncation=True,
        padding="max_length",        # фикс: выравниваем каждый chunk до MAX_LENGTH
        max_length=MAX_LENGTH,
        stride=STRIDE,
        return_overflowing_tokens=True,
    )

    input_ids_chunks = encoded["input_ids"][:MAX_CHUNKS]
    attention_mask_chunks = encoded["attention_mask"][:MAX_CHUNKS]

    n_chunks = len(input_ids_chunks)

    if n_chunks < MAX_CHUNKS:
        pad_len = MAX_CHUNKS - n_chunks
        pad_ids = [tokenizer.pad_token_id] * MAX_LENGTH
        pad_mask = [0] * MAX_LENGTH
        input_ids_chunks += [pad_ids] * pad_len
        attention_mask_chunks += [pad_mask] * pad_len

    return {
        "input_ids": input_ids_chunks,
        "attention_mask": attention_mask_chunks,
        "labels": example["label_id"],
        "num_chunks": n_chunks,
    }


class ChunkMeanPoolRobertaClassifier(nn.Module):
    def __init__(self, model_name, num_labels, id2label=None, label2id=None, class_weights=None):
        super().__init__()
        self.config = RobertaConfig.from_pretrained(
            model_name,
            num_labels=num_labels,
            id2label=id2label,
            label2id=label2id,
            output_hidden_states=False,
        )
        self.roberta = RobertaModel.from_pretrained(model_name, config=self.config)
        self.roberta.gradient_checkpointing_enable()

        hidden_size = self.config.hidden_size

        self.dropout1 = nn.Dropout(HEAD_DROPOUT)
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.act = nn.GELU()
        self.dropout2 = nn.Dropout(HEAD_DROPOUT)
        self.classifier = nn.Linear(hidden_size, num_labels)

        if class_weights is not None:
            self.register_buffer("class_weights", class_weights)
        else:
            self.class_weights = None

    def token_mean_pool(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        masked = last_hidden_state * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def chunk_mean_pool(self, chunk_embeddings, chunk_mask):
        mask = chunk_mask.unsqueeze(-1).type_as(chunk_embeddings)
        masked = chunk_embeddings * mask
        summed = masked.sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1e-9)
        return summed / counts

    def forward(self, input_ids=None, attention_mask=None, labels=None, num_chunks=None, **kwargs):
        # input_ids: [batch, chunks, seq]
        batch_size, n_chunks, seq_len = input_ids.shape

        flat_input_ids = input_ids.view(batch_size * n_chunks, seq_len)
        flat_attention_mask = attention_mask.view(batch_size * n_chunks, seq_len)

        outputs = self.roberta(
            input_ids=flat_input_ids,
            attention_mask=flat_attention_mask,
            return_dict=True,
        )

        token_pooled = self.token_mean_pool(outputs.last_hidden_state, flat_attention_mask)
        chunk_embeddings = token_pooled.view(batch_size, n_chunks, -1)

        if num_chunks is None:
            chunk_mask = (attention_mask.sum(dim=-1) > 0).long()
        else:
            arange = torch.arange(n_chunks, device=input_ids.device).unsqueeze(0)
            chunk_mask = (arange < num_chunks.unsqueeze(1)).long()

        doc_embedding = self.chunk_mean_pool(chunk_embeddings, chunk_mask)

        x = self.dropout1(doc_embedding)
        x = self.fc1(x)
        x = self.act(x)
        x = self.dropout2(x)
        logits = self.classifier(x)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(
                weight=self.class_weights,
                label_smoothing=LABEL_SMOOTHING
            )
            loss = loss_fct(logits.view(-1, self.config.num_labels), labels.view(-1))

        return {"loss": loss, "logits": logits} if loss is not None else {"logits": logits}


class ChunkDataCollator:
    def __call__(self, features):
        # Дополнительные проверки на всякий случай
        for i, f in enumerate(features):
            assert len(f["input_ids"]) == MAX_CHUNKS, f"Bad num_chunks in sample {i}"
            assert all(len(x) == MAX_LENGTH for x in f["input_ids"]), f"Bad input_ids len in sample {i}"
            assert all(len(x) == MAX_LENGTH for x in f["attention_mask"]), f"Bad attention_mask len in sample {i}"

        input_ids = torch.tensor([f["input_ids"] for f in features], dtype=torch.long)
        attention_mask = torch.tensor([f["attention_mask"] for f in features], dtype=torch.long)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        num_chunks = torch.tensor([f["num_chunks"] for f in features], dtype=torch.long)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "num_chunks": num_chunks,
        }


class WeightedChunkTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        outputs = model(**inputs)
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        return (loss, outputs) if return_outputs else loss

    def create_optimizer(self):
        if self.optimizer is None:
            decay_parameters = self.get_decay_parameter_names(self.model)

            encoder_params_decay = []
            encoder_params_no_decay = []
            head_params_decay = []
            head_params_no_decay = []

            for name, param in self.model.named_parameters():
                if not param.requires_grad:
                    continue

                is_decay = name in decay_parameters
                is_encoder = name.startswith("roberta.")

                if is_encoder and is_decay:
                    encoder_params_decay.append(param)
                elif is_encoder and not is_decay:
                    encoder_params_no_decay.append(param)
                elif not is_encoder and is_decay:
                    head_params_decay.append(param)
                else:
                    head_params_no_decay.append(param)

            optimizer_grouped_parameters = [
                {
                    "params": encoder_params_decay,
                    "weight_decay": WEIGHT_DECAY,
                    "lr": LR_ENCODER,
                },
                {
                    "params": encoder_params_no_decay,
                    "weight_decay": 0.0,
                    "lr": LR_ENCODER,
                },
                {
                    "params": head_params_decay,
                    "weight_decay": WEIGHT_DECAY,
                    "lr": LR_HEAD,
                },
                {
                    "params": head_params_no_decay,
                    "weight_decay": 0.0,
                    "lr": LR_HEAD,
                },
            ]

            self.optimizer = torch.optim.AdamW(
                optimizer_grouped_parameters,
                betas=(0.9, 0.999),
                eps=1e-8,
            )

        return self.optimizer

In [10]:
TRAIN_FILE = os.path.join(drive_root, 'train_augmented.csv')
TEST_FILE = os.path.join(drive_root, 'test.csv')

for var_name in [
    "trainer",
    "train_df",
    "test_df",
    "dataset",
    "eval_metrics",
    "labels",
    "label2id",
    "id2label",
    "class_weights",
]:
    if var_name in globals():
        del globals()[var_name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

train_df = pd.read_csv(TRAIN_FILE)
test_df = pd.read_csv(TEST_FILE)

train_df = train_df[[TEXT_COL, LABEL_COL]].copy().dropna()
test_df = test_df[[TEXT_COL, LABEL_COL]].copy().dropna()

train_df[TEXT_COL] = train_df[TEXT_COL].astype(str).str.strip()
train_df[LABEL_COL] = train_df[LABEL_COL].astype(str).str.strip()

test_df[TEXT_COL] = test_df[TEXT_COL].astype(str).str.strip()
test_df[LABEL_COL] = test_df[LABEL_COL].astype(str).str.strip()

train_df = train_df[(train_df[TEXT_COL] != "") & (train_df[LABEL_COL] != "")].reset_index(drop=True)
test_df = test_df[(test_df[TEXT_COL] != "") & (test_df[LABEL_COL] != "")].reset_index(drop=True)

labels = sorted(train_df[LABEL_COL].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

unknown_test_labels = sorted(set(test_df[LABEL_COL].unique()) - set(labels))
if unknown_test_labels:
    raise ValueError(f"В TEST_FILE есть unseen labels: {unknown_test_labels}")

train_df["label_id"] = train_df[LABEL_COL].map(label2id)
test_df["label_id"] = test_df[LABEL_COL].map(label2id)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(labels)),
    y=train_df["label_id"].values
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df[[TEXT_COL, "label_id"]], preserve_index=False),
    "validation": Dataset.from_pandas(test_df[[TEXT_COL, "label_id"]], preserve_index=False)
})

dataset = dataset.map(tokenize_document)
dataset.set_format(type="python")

data_collator = ChunkDataCollator()

def model_init():
    return ChunkMeanPoolRobertaClassifier(
        model_name=MODEL_NAME,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
        class_weights=class_weights_tensor,
    )

training_args = TrainingArguments(
    output_dir="./ruroberta_chunk_meanpool_best",
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR_ENCODER,
    lr_scheduler_type="cosine",
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    fp16=torch.cuda.is_available(),
    seed=SEED,
    dataloader_num_workers=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    max_grad_norm=1.0,
    remove_unused_columns=False,
    disable_tqdm=False,
)

trainer = WeightedChunkTrainer(
    model_init=model_init,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
eval_metrics = trainer.evaluate(dataset["validation"])

print("\nFINAL METRICS")
print(f"balanced_accuracy: {eval_metrics['eval_balanced_accuracy']:.6f}")
print(f"f1_macro:          {eval_metrics['eval_f1_macro']:.6f}")

Map:   0%|          | 0/1826 [00:00<?, ? examples/s]

Map:   0%|          | 0/355 [00:00<?, ? examples/s]

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ai-forever/ruRoberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: ai-forever/ruRoberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Epoch,Training Loss,Validation Loss,Balanced Accuracy,F1 Macro
1,30.034981,3.639333,0.027778,0.006511
2,29.476865,3.581292,0.028382,0.007685
3,28.755864,3.532710,0.027778,0.006495
4,27.147062,3.447191,0.044123,0.020416
5,23.798668,2.861836,0.145227,0.114756
6,19.448995,2.601081,0.189091,0.169752
7,16.558257,2.465682,0.280577,0.262980
8,14.710697,2.406157,0.309547,0.296722
9,13.676632,2.385685,0.307967,0.294318
10,13.253026,2.389532,0.307967,0.294657



FINAL METRICS
balanced_accuracy: 0.309547
f1_macro:          0.296722


In [11]:
import os
import json
import torch
from pathlib import Path

SAVE_DIR = os.path.join(drive_root, "saved_models", "ruroberta_chunk_meanpool_best")

def save_best_model_for_reuse(trainer, tokenizer, save_dir, label2id, id2label, extra_config=None):
    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)

    model = trainer.model
    model.eval()

    # Берём state_dict и выкидываем class_weights, чтобы не таскать лишний buffer
    state_dict = model.state_dict()
    filtered_state_dict = {k: v for k, v in state_dict.items() if k != "class_weights"}

    torch.save(filtered_state_dict, save_path / "pytorch_model.bin")
    tokenizer.save_pretrained(save_path)

    training_meta = {
        "model_name": MODEL_NAME,
        "max_length": MAX_LENGTH,
        "stride": STRIDE,
        "max_chunks": MAX_CHUNKS,
        "num_labels": len(label2id),
        "label2id": label2id,
        "id2label": {str(k): v for k, v in id2label.items()},
        "text_col": TEXT_COL,
        "label_col": LABEL_COL,
        "head_dropout": HEAD_DROPOUT,
        "label_smoothing": LABEL_SMOOTHING,
    }

    if extra_config is not None:
        training_meta.update(extra_config)

    with open(save_path / "training_meta.json", "w", encoding="utf-8") as f:
        json.dump(training_meta, f, ensure_ascii=False, indent=2)

    print(f"Model weights saved to: {save_path / 'pytorch_model.bin'}")
    print(f"Tokenizer saved to:     {save_path}")
    print(f"Meta saved to:          {save_path / 'training_meta.json'}")
    print("Excluded from checkpoint: ['class_weights']")


save_best_model_for_reuse(
    trainer=trainer,
    tokenizer=tokenizer,
    save_dir=SAVE_DIR,
    label2id=label2id,
    id2label=id2label,
    extra_config={
        "train_file": TRAIN_FILE,
        "test_file": TEST_FILE,
    }
)

Model weights saved to: /content/drive/MyDrive/papadyk-collab/vkr/saved_models/ruroberta_chunk_meanpool_best/pytorch_model.bin
Tokenizer saved to:     /content/drive/MyDrive/papadyk-collab/vkr/saved_models/ruroberta_chunk_meanpool_best
Meta saved to:          /content/drive/MyDrive/papadyk-collab/vkr/saved_models/ruroberta_chunk_meanpool_best/training_meta.json
